# NB05: Geospatial Analysis

## Welfare Scheme Participation Analysis

Choropleth maps + spatial autocorrelation (Moran's I, LISA) to identify geographic clusters of scheme gaps.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.config import MASTER_DATA, MAPS_DIR
from src.feature_engineer import build_features
from src.geospatial import (
    load_geodata, merge_with_geodata, compute_morans_i,
    compute_lisa, classify_lisa, create_choropleth, create_multi_layer_map
)
from src.visualization import setup_plot_style, save_figure

setup_plot_style()
master = pd.read_csv(MASTER_DATA)
df = build_features(master)
print(f'Dataset: {df.shape[0]} districts')

## 1. Load & Merge GeoJSON

In [ ]:
gdf = load_geodata()
print(f'GeoJSON: {len(gdf)} polygons, columns: {list(gdf.columns)}')
merged = merge_with_geodata(gdf, df)
n_matched = merged[merged['population_total'].notna()].shape[0]
print(f'Matched districts: {n_matched}/{len(merged)}')

## 2. Choropleth Maps

In [ ]:
m = create_choropleth(
    merged, 'mgnrega_gap_pct',
    'MGNREGA Participation Gap %', 'Gap %',
    cmap='YlOrRd',
    save_path=f'{MAPS_DIR}/mgnrega_gap_choropleth.html'
)
print('Saved MGNREGA choropleth')

In [ ]:
m = create_choropleth(
    merged, 'health_insurance_gap_pct',
    'Health Insurance Coverage Gap %', 'Gap %',
    cmap='BuPu',
    save_path=f'{MAPS_DIR}/health_ins_gap_choropleth.html'
)
print('Saved health insurance choropleth')

In [ ]:
m = create_choropleth(
    merged, 'combined_gap_score',
    'Combined Welfare Gap Score', 'Score',
    cmap='RdYlGn_r',
    save_path=f'{MAPS_DIR}/combined_gap_choropleth.html'
)
print('Saved combined gap choropleth')

## 3. Multi-Layer Interactive Map

In [ ]:
m = create_multi_layer_map(
    merged,
    save_path=f'{MAPS_DIR}/multi_layer_welfare_map.html'
)
print('Saved multi-layer map')

## 4. Spatial Autocorrelation (Moran's I)

In [ ]:
from src.data_cleaner import normalize_district_name

merged_valid = merged.dropna(subset=['mgnrega_gap_pct']).copy()
print(f'Valid polygons for Moran: {len(merged_valid)}')

In [ ]:
try:
    mi_mgnrega = compute_morans_i(merged_valid, 'mgnrega_gap_pct')
    print('MGNREGA Gap Moran I:')
    for k, v in mi_mgnrega.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'Moran I computation failed: {e}')

In [ ]:
try:
    mi_health = compute_morans_i(merged_valid, 'health_insurance_gap_pct')
    print('Health Insurance Gap Moran I:')
    for k, v in mi_health.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'Moran I computation failed: {e}')

## 5. LISA Cluster Analysis

In [ ]:
try:
    lisa = compute_lisa(merged_valid, 'combined_gap_score')
    lisa_labels = classify_lisa(lisa)
    lisa_counts = pd.Series(lisa_labels).value_counts()
    print('LISA Cluster Counts:')
    print(lisa_counts.to_string())
except Exception as e:
    print(f'LISA computation failed: {e}')
    lisa_labels = None

## 6. Static Visualization of Gap Patterns

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

merged.plot(column='mgnrega_gap_pct', cmap='YlOrRd', legend=True, ax=axes[0],
            missing_kwds={'color': 'lightgrey'}, edgecolor='black', linewidth=0.2)
axes[0].set_title('MGNREGA Participation Gap %', fontsize=14)
axes[0].axis('off')

merged.plot(column='health_insurance_gap_pct', cmap='BuPu', legend=True, ax=axes[1],
            missing_kwds={'color': 'lightgrey'}, edgecolor='black', linewidth=0.2)
axes[1].set_title('Health Insurance Coverage Gap %', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
save_figure(fig, 'geospatial_gap_maps')
plt.show()